# Understanding scan grouping in QSIPrep

Before QSIPrep processes a single voxel, it has to answer three questions about
your diffusion data:

1. **Which scans estimate each fieldmap?** (in BIDS terms: `B0FieldIdentifier`)
2. **Which fieldmap corrects which scan?** (`B0FieldSource`)
3. **Which scans are combined into one preprocessed output file?** (`MultipartID`)

If you set those three fields in your sidecar JSONs, QSIPrep uses your answers
verbatim. Wherever you did not, it infers an answer and *tells you it guessed* —
every decision carries a provenance tag: **curated** (you set it),
**cli-override** (a command-line flag), **inferred** (a heuristic), or
**IntendedFor** (translated from the deprecated fieldmap linkage — see the
appendix).

This notebook builds tiny fake datasets *in memory* — no files, no BIDS layout,
just filenames and sidecar values — and shows how each metadata change alters
the grouping. Nothing is processed; grouping only reads metadata.


In [1]:
import html as _html

from IPython.display import HTML, display

from qsiprep.grouping import describe_processing, render_html, report_text
from qsiprep.grouping.inference import build_grouping
from qsiprep.grouping.models import DistortionSignature, FileRecord

SUBJECT = '/bids/sub-01'


def scan(
    name,
    folder='dwi',
    suffix=None,
    pe_dir='j-',
    readout=0.05,
    shim=None,
    session=None,
    b0field_id=(),
    b0field_source=(),
    multipart_id=None,
    intended_for=(),
):
    """One imaging file plus the sidecar fields that grouping reads."""
    suffix = suffix or ('dwi' if folder == 'dwi' else 'epi')
    sess = f'ses-{session}/' if session else ''
    as_tuple = lambda v: (v,) if isinstance(v, str) else tuple(v)  # noqa: E731
    return FileRecord(
        path=f'{SUBJECT}/{sess}{folder}/{name}',
        datatype='anat' if suffix in ('T1w', 'T2w') else folder,
        suffix=suffix,
        session=session,
        signature=DistortionSignature(
            pe_dir=pe_dir, readout_time=readout, shim=tuple(shim) if shim else None
        ),
        b0field_identifiers=as_tuple(b0field_id),
        b0field_sources=as_tuple(b0field_source),
        multipart_id=multipart_id,
        intended_for=as_tuple(intended_for),
    )


def group(*records, **options):
    """Run the full grouping on in-memory records."""
    return build_grouping(list(records), subject_id='01', **options)


def show(grouping, height=560, backend='fsl'):
    """Display the explanatory HTML page for a grouping inline."""
    page = _html.escape(render_html(grouping, backend=backend))
    display(
        HTML(
            f'<iframe srcdoc="{page}" style="width:100%;height:{height}px;'
            'border:1px solid #cbd5e1;border-radius:8px;background:#fff"></iframe>'
        )
    )

## 1. One scan, no metadata

The simplest possible subject: a single DWI series, no fieldmaps, no curation.
Even here grouping has decisions to report: the scan forms one *distortion
group* (files that share the same susceptibility distortion), it becomes one
output file, and — since there is nothing to correct it with — susceptibility
distortion correction (SDC) is skipped.


In [2]:
g = group(scan('sub-01_dir-AP_dwi.nii.gz'))
print(report_text(g))

DWI grouping for sub-01

Output "sub-01_dir-AP" (MultipartID auto+concat+0 [inferred]): 1 series
  Distortion group sub-01_dir-AP (PE j-, TRT 0.05s):
    - sub-01_dir-AP_dwi.nii.gz
    corrected by: nothing (no fieldmap found)

Fieldmap estimations: none found.



In [3]:
show(g, height=620)

Notice the *why* lines. The output name carries `MultipartID auto+concat+0
[inferred]`: the `auto+` prefix is reserved for identifiers QSIPrep made up
(`+` cannot appear in curated BIDS values, so they can never collide with your
names).

## 2. A reverse phase-encoded partner appears

Now add a second series acquired with the opposite phase encoding (`j` instead
of `j-` — an AP/PA pair). Two things happen:

- The scans land in **different distortion groups**, because they are squished
  in opposite directions along the phase-encoding axis. A distortion group is
  defined by `PhaseEncodingDirection` + `TotalReadoutTime` + `ShimSetting`.
- QSIPrep notices the opposite polarities and **infers a PEPOLAR fieldmap
  estimation** from the pair: their b=0 images can be compared to measure the
  distortion field. Both groups are corrected by it.

This is the zero-curation "HCP-style" case: acquire blip-up and blip-down, get
susceptibility correction for free.


In [4]:
ap = scan('sub-01_dir-AP_dwi.nii.gz', pe_dir='j-')
pa = scan('sub-01_dir-PA_dwi.nii.gz', pe_dir='j')
g = group(ap, pa)
show(g, height=780)

The estimation card is **amber (inferred)** and named `auto+pepolar+j` — again
the reserved `auto+` prefix. The blip diagram (&#9650; j &harr; &#9660; j&minus;)
shows the pairing, and each scan row carries the Ⓐ chip marking that its b=0
volumes feed estimation A.

## 3. Curating fieldmaps: `B0FieldIdentifier` / `B0FieldSource`

Guessing is fine until it isn't. The BIDS way to make fieldmap relationships
explicit is a pair of sidecar fields:

- **`B0FieldIdentifier`** names an estimation, and goes on every file whose
  data *feed* it.
- **`B0FieldSource`** goes on each DWI series and names the estimation that
  should *correct* it.

Here is the same AP/PA pair from section 2, curated. The data are physically
identical — only the sidecars changed. The estimation now has **your** name
(no `auto+`), everything renders green, and no heuristic was involved:


In [5]:
ap = scan(
    'sub-01_dir-AP_dwi.nii.gz',
    pe_dir='j-',
    b0field_id='pepolar_fmap',
    b0field_source='pepolar_fmap',
)
pa = scan(
    'sub-01_dir-PA_dwi.nii.gz',
    pe_dir='j',
    b0field_id='pepolar_fmap',
    b0field_source='pepolar_fmap',
)
g = group(ap, pa)
show(g, height=780)

A dedicated `fmap/` image works exactly the same way: put the
`B0FieldIdentifier` on the fieldmap file *and* on the DWI (its b=0 supplies
the opposite blip in the estimation), and point the DWI's `B0FieldSource` at
it:


In [6]:
dwi = scan(
    'sub-01_dir-AP_dwi.nii.gz',
    pe_dir='j-',
    b0field_id='epi_fmap',
    b0field_source='epi_fmap',
)
fmap = scan(
    'sub-01_dir-PA_epi.nii.gz',
    folder='fmap',
    pe_dir='j',
    b0field_id='epi_fmap',
)
g = group(dwi, fmap)
show(g, height=700)

The fieldmap file lives in `fmap/`, so it is listed on the estimation card
("from `fmap/`") but never appears inside an output box — fieldmaps measure
distortion; they are not part of your diffusion data.

(Older datasets link fieldmaps with the `IntendedFor` field instead. QSIPrep
still honors it, but it is deprecated — see the appendix at the end of this
notebook.)

## 4. Splitting outputs with `MultipartID` — and borrowing

By default, all series in a session with compatible distortion parameters are
concatenated into **one** output file (better head-motion correction, one file
to analyze). Four runs, one output:


In [7]:
runs = [
    scan('sub-01_dir-AP_run-1_dwi.nii.gz', pe_dir='j-'),
    scan('sub-01_dir-AP_run-2_dwi.nii.gz', pe_dir='j-'),
    scan('sub-01_dir-PA_run-1_dwi.nii.gz', pe_dir='j'),
    scan('sub-01_dir-PA_run-2_dwi.nii.gz', pe_dir='j'),
]
g = group(*runs)
print(report_text(g))

DWI grouping for sub-01

Output "sub-01" (MultipartID auto+concat+0 [inferred]): 4 series
  Distortion group sub-01_dir-AP (PE j-, TRT 0.05s):
    - sub-01_dir-AP_run-1_dwi.nii.gz
    - sub-01_dir-AP_run-2_dwi.nii.gz
    corrected by: auto+pepolar+j [inferred]
  Distortion group sub-01_dir-PA (PE j, TRT 0.05s):
    - sub-01_dir-PA_run-1_dwi.nii.gz
    - sub-01_dir-PA_run-2_dwi.nii.gz
    corrected by: auto+pepolar+j [inferred]

Fieldmap estimations:
  auto+pepolar+j [inferred]: PEPOLAR (reverse phase-encoding)
    - sub-01_dir-AP_run-1_dwi.nii.gz
    - sub-01_dir-AP_run-2_dwi.nii.gz
    - sub-01_dir-PA_run-1_dwi.nii.gz
    - sub-01_dir-PA_run-2_dwi.nii.gz
    phase encoding axes: j (bidirectional)



`MultipartID` is the sidecar field that controls this. Give run-1 and run-2
different values and you get two output files. But watch what happens to the
fieldmap estimation — **estimation membership and concatenation membership are
independent by design**. All four scans still feed the *same* PEPOLAR
estimation; each output then *borrows* the b=0 images of the scans that ended
up in the other output:


In [8]:
runs = [
    scan('sub-01_dir-AP_run-1_dwi.nii.gz', pe_dir='j-', multipart_id='part1'),
    scan('sub-01_dir-AP_run-2_dwi.nii.gz', pe_dir='j-', multipart_id='part2'),
    scan('sub-01_dir-PA_run-1_dwi.nii.gz', pe_dir='j', multipart_id='part1'),
    scan('sub-01_dir-PA_run-2_dwi.nii.gz', pe_dir='j', multipart_id='part2'),
]
g = group(*runs)
show(g, height=900)

The dashed "borrows b=0 volumes from…" sentences make this visible: borrowed
scans improve the *fieldmap*, but their diffusion volumes are **not** written
into that output file.

## 5. Partial curation: curating some scans and not others

What if you curate *some* scans and leave the rest alone? QSIPrep treats
curation as a **wall, not a hint**: scans already linked to a fieldmap by
`B0FieldIdentifier`/`B0FieldSource` are never mixed with unlinked scans,
because a curator who linked some scans and not others presumably did not
want them combined. Every partial-curation pattern also draws a warning
telling you exactly which field to finish curating.

**Partial `B0FieldIdentifier`** — run-1 is curated into `pepolar01`, run-2
carries nothing. Run-2 still gets an automatic PEPOLAR estimation, but it is
built *only* from the uncurated scans — the curated pair is off limits:


In [9]:
runs = [
    scan(
        'sub-01_dir-AP_run-1_dwi.nii.gz',
        pe_dir='j-',
        b0field_id='pepolar01',
        b0field_source='pepolar01',
    ),
    scan(
        'sub-01_dir-PA_run-1_dwi.nii.gz',
        pe_dir='j',
        b0field_id='pepolar01',
        b0field_source='pepolar01',
    ),
    scan('sub-01_dir-AP_run-2_dwi.nii.gz', pe_dir='j-'),
    scan('sub-01_dir-PA_run-2_dwi.nii.gz', pe_dir='j'),
]
g = group(*runs)
show(g, height=900)

Two estimation cards — curated Ⓑ (green) and inferred Ⓐ (amber) — and no scan
appears on both. The `mixed-application-provenance` warning nudges you to
finish the job.

**A stranded scan** — the sharpest version of the wall. Here the only reverse
phase-encoded partners of the uncurated run-2 AP scan are curated, so it gets
*no* PEPOLAR estimation at all (and, with no T2w to fall back on, no
correction), with a warning explaining why:


In [10]:
runs = [
    scan(
        'sub-01_dir-AP_run-1_dwi.nii.gz',
        pe_dir='j-',
        b0field_id='pepolar01',
        b0field_source='pepolar01',
    ),
    scan(
        'sub-01_dir-PA_run-1_dwi.nii.gz',
        pe_dir='j',
        b0field_id='pepolar01',
        b0field_source='pepolar01',
    ),
    scan('sub-01_dir-AP_run-2_dwi.nii.gz', pe_dir='j-'),
]
g = group(*runs)
print(report_text(g))

DWI grouping for sub-01

Output "sub-01" (MultipartID auto+concat+0 [inferred]): 3 series
  Distortion group sub-01_dir-AP_run-1 (PE j-, TRT 0.05s):
    - sub-01_dir-AP_run-1_dwi.nii.gz
    corrected by: pepolar01 [curated]
  Distortion group sub-01_dir-AP_run-2 (PE j-, TRT 0.05s):
    - sub-01_dir-AP_run-2_dwi.nii.gz
    corrected by: nothing (no fieldmap found)
  Distortion group sub-01_dir-PA_run-1 (PE j, TRT 0.05s):
    - sub-01_dir-PA_run-1_dwi.nii.gz
    corrected by: pepolar01 [curated]

Fieldmap estimations:
  pepolar01 [curated]: PEPOLAR (reverse phase-encoding)
    - sub-01_dir-AP_run-1_dwi.nii.gz
    - sub-01_dir-PA_run-1_dwi.nii.gz
    phase encoding axes: j (bidirectional)

Notes:
  WARNING [reverse-pe-not-pooled]: sub-01_dir-AP_run-2_dwi.nii.gz: the only reverse phase-encoded series on this axis are already linked to fieldmaps by curated metadata (B0FieldIdentifier/B0FieldSource or IntendedFor), and linked series are never pooled with unlinked ones. Curate the metadata on

**Partial `MultipartID`** works the same way: series carrying one are combined
as asked, and series without one are *not* combined with anything — each
becomes its own output file:


In [11]:
runs = [
    scan('sub-01_dir-AP_run-1_dwi.nii.gz', pe_dir='j-', multipart_id='combined'),
    scan('sub-01_dir-PA_run-1_dwi.nii.gz', pe_dir='j', multipart_id='combined'),
    scan('sub-01_dir-AP_run-2_dwi.nii.gz', pe_dir='j-'),
    scan('sub-01_dir-PA_run-2_dwi.nii.gz', pe_dir='j'),
]
g = group(*runs)
for concat in g.concatenation_groups.values():
    print(f'{concat.output_name}: {len(concat.dwi_files)} scan(s)')
print()
for issue in g.warnings:
    print(issue.render())

sub-01_dir-AP_run-2: 1 scan(s)
sub-01_dir-PA_run-2: 1 scan(s)
sub-01_run-1: 2 scan(s)

WARNING [partial-multipart]: 2 DWI series have no MultipartID while other series in this subject do. Series without one are NOT combined with anything: each becomes its own output. Set MultipartID on every series (or on none) to control concatenation explicitly.
WARNING [estimation-spans-outputs]: Fieldmap estimation 'auto+pepolar+j' corrects DWI series in 3 different outputs; it will be estimated once per output.


Note that the *estimation* still pools all four scans (none of them carries
`B0Field*` curation, and estimation membership is independent of
concatenation) — so each single-scan output borrows the others' b=0 images
for its fieldmap.

## 6. `ShimSetting`: when the scanner re-shims

The scanner's shim state is part of the distortion signature. If run-2 was
acquired after a re-shim (different `ShimSetting` values), its distortion no
longer matches run-1 — so QSIPrep refuses to treat them as interchangeable:
the runs split into separate outputs and separate estimation pools, with a
warning explaining why.


In [12]:
runs = [
    scan('sub-01_dir-AP_run-1_dwi.nii.gz', pe_dir='j-', shim=(1.0, 2.0, 3.0)),
    scan('sub-01_dir-PA_run-1_dwi.nii.gz', pe_dir='j', shim=(1.0, 2.0, 3.0)),
    scan('sub-01_dir-AP_run-2_dwi.nii.gz', pe_dir='j-', shim=(9.0, 9.0, 9.0)),
    scan('sub-01_dir-PA_run-2_dwi.nii.gz', pe_dir='j', shim=(9.0, 9.0, 9.0)),
]
g = group(*runs)
print(report_text(g))

DWI grouping for sub-01

Output "sub-01_run-1" (MultipartID auto+concat+0 [inferred]): 2 series
  Distortion group sub-01_dir-AP_run-1 (PE j-, TRT 0.05s, shimmed):
    - sub-01_dir-AP_run-1_dwi.nii.gz
    corrected by: auto+pepolar+shim1+j [inferred]
  Distortion group sub-01_dir-PA_run-1 (PE j, TRT 0.05s, shimmed):
    - sub-01_dir-PA_run-1_dwi.nii.gz
    corrected by: auto+pepolar+shim1+j [inferred]

Output "sub-01_run-2" (MultipartID auto+concat+1 [inferred]): 2 series
  Distortion group sub-01_dir-AP_run-2 (PE j-, TRT 0.05s, shimmed):
    - sub-01_dir-AP_run-2_dwi.nii.gz
    corrected by: auto+pepolar+shim2+j [inferred]
  Distortion group sub-01_dir-PA_run-2 (PE j, TRT 0.05s, shimmed):
    - sub-01_dir-PA_run-2_dwi.nii.gz
    corrected by: auto+pepolar+shim2+j [inferred]

Fieldmap estimations:
  auto+pepolar+shim1+j [inferred]: PEPOLAR (reverse phase-encoding)
    - sub-01_dir-AP_run-1_dwi.nii.gz
    - sub-01_dir-PA_run-1_dwi.nii.gz
    phase encoding axes: j (bidirectional)
  auto

If you know the re-shim was inconsequential (or you want cross-shim correction
anyway), `--ignore shims` treats all shim values as compatible:


In [13]:
g = group(*runs, ignore_shims=True)
print(report_text(g))

DWI grouping for sub-01

Output "sub-01" (MultipartID auto+concat+0 [inferred]): 4 series
  Distortion group sub-01_dir-AP_run-1 (PE j-, TRT 0.05s, shimmed):
    - sub-01_dir-AP_run-1_dwi.nii.gz
    corrected by: auto+pepolar+j [inferred]
  Distortion group sub-01_dir-AP_run-2 (PE j-, TRT 0.05s, shimmed):
    - sub-01_dir-AP_run-2_dwi.nii.gz
    corrected by: auto+pepolar+j [inferred]
  Distortion group sub-01_dir-PA_run-1 (PE j, TRT 0.05s, shimmed):
    - sub-01_dir-PA_run-1_dwi.nii.gz
    corrected by: auto+pepolar+j [inferred]
  Distortion group sub-01_dir-PA_run-2 (PE j, TRT 0.05s, shimmed):
    - sub-01_dir-PA_run-2_dwi.nii.gz
    corrected by: auto+pepolar+j [inferred]

Fieldmap estimations:
  auto+pepolar+j [inferred]: PEPOLAR (reverse phase-encoding)
    - sub-01_dir-AP_run-1_dwi.nii.gz
    - sub-01_dir-AP_run-2_dwi.nii.gz
    - sub-01_dir-PA_run-1_dwi.nii.gz
    - sub-01_dir-PA_run-2_dwi.nii.gz
    phase encoding axes: j (bidirectional)

Notes:
  WARNING [shims-ignored]: 2 dif

## 7. No fieldmap at all: the fieldmap-less ladder

With no fieldmap and no reverse-PE partner, QSIPrep walks a ladder of
fallbacks. If the subject has a T2w image, it infers a **T2w registration**
(T2Wreg) correction — the distorted b=0 is registered to the undistorted T2w:


In [14]:
dwi = scan('sub-01_dir-AP_dwi.nii.gz', pe_dir='j-')
t1w = scan('sub-01_T1w.nii.gz', folder='anat', suffix='T1w', pe_dir=None, readout=None)
t2w = scan('sub-01_T2w.nii.gz', folder='anat', suffix='T2w', pe_dir=None, readout=None)
g = group(dwi, t1w, t2w)
show(g, height=680, backend='tortoise')

Command-line flags climb the ladder explicitly (provenance **cli-override**,
purple): `--use-synb0` synthesizes an undistorted b=0 from the T1w, and the
niworkflows SyN-SDC registers an inverted T1w to a fieldmap atlas. A real
fieldmap always outranks every fieldmap-less method — the flags only apply to
series that would otherwise go uncorrected:


In [15]:
g = group(dwi, t1w, t2w, use_synb0=True)
print(report_text(g))

DWI grouping for sub-01

Output "sub-01_dir-AP" (MultipartID auto+concat+0 [inferred]): 1 series
  Distortion group sub-01_dir-AP (PE j-, TRT 0.05s):
    - sub-01_dir-AP_dwi.nii.gz
    corrected by: auto+synb0 [cli-override]

Fieldmap estimations:
  auto+synb0 [cli-override]: SyNb0 synthetic b=0
    - sub-01_T1w.nii.gz



## 8. Same grouping, different pipelines

The grouping describes your *data*. How a processing backend consumes it is a
separate decision — and the previews spell out the difference. The FSL path
pools all b=0 images into one TOPUP estimation and models everything jointly in
eddy; the TORTOISE path motion-corrects each distortion group separately and
feeds blip-up/blip-down to DRBUDDI; the two-stage path runs TOPUP+eddy first
and then refines with DRBUDDI.


In [16]:
runs = [
    scan('sub-01_dir-AP_dwi.nii.gz', pe_dir='j-'),
    scan('sub-01_dir-PA_dwi.nii.gz', pe_dir='j'),
]
g = group(*runs)
for backend in ('fsl', 'tortoise', 'mixed'):
    print(describe_processing(g, backend))

Processing preview: FSL path (TOPUP fieldmap estimation + eddy HMC/SDC)
-----------------------------------------------------------------------

Output "sub-01" (2 series):
  1. Each series is denoised on its own, then all 2 series are concatenated. (--denoise-after-combining reverses this order.)
  2. TOPUP estimates the susceptibility field from b=0 images spanning 2 distortion groups (2 rows in the acquisition-parameters file).
  3. eddy corrects head motion, eddy currents, and susceptibility distortion in one model, using the TOPUP field. Every volume is assigned to its distortion group in the eddy index file.
  4. The corrected series is written as one output file.

Processing preview: TORTOISE path (DIFFPREP HMC + DRBUDDI SDC)
--------------------------------------------------------------

Output "sub-01" (2 series):
  1. Each series is denoised on its own, then all 2 series are concatenated. (--denoise-after-combining reverses this order.)
  2. DIFFPREP corrects head motion and 

The HTML page shows the same information: the chosen backend's steps are
expanded on each output box, with the alternatives collapsed underneath
("if run with the … workflow instead").

## Appendix: `IntendedFor` (deprecated)

Older datasets link fieldmaps to DWI series with the `IntendedFor` field on
the fieldmap's sidecar. QSIPrep still honors it — the linkage is translated
into an estimation with provenance **IntendedFor** (blue) — but it is a
legacy mechanism. **Do not use it for new curation**; use
`B0FieldIdentifier`/`B0FieldSource` (section 3) instead.


In [17]:
dwi = scan('sub-01_dir-AP_dwi.nii.gz', pe_dir='j-')
fmap = scan(
    'sub-01_dir-PA_epi.nii.gz',
    folder='fmap',
    pe_dir='j',
    intended_for=dwi.path,
)
g = group(dwi, fmap)
show(g, height=700)

Three rules keep the legacy path predictable:

**1. `B0Field*` supersedes it.** If a fieldmap carries *both* `IntendedFor`
and `B0FieldIdentifier`, the `B0Field*` links are used exclusively and the
`IntendedFor` is ignored, with a warning:


In [18]:
dwi = scan(
    'sub-01_dir-AP_dwi.nii.gz',
    pe_dir='j-',
    b0field_id='my_fmap',  # its b=0 volumes feed the estimation...
    b0field_source='my_fmap',  # ...and the estimation corrects it
)
fmap = scan(
    'sub-01_dir-PA_epi.nii.gz',
    folder='fmap',
    pe_dir='j',
    b0field_id='my_fmap',
    intended_for=dwi.path,  # ignored: B0FieldIdentifier supersedes it
)
g = group(dwi, fmap)
for issue in g.issues:
    print(issue.render())

WARNING [intendedfor-superseded]: sub-01_dir-PA_epi.nii.gz carries both B0FieldIdentifier and IntendedFor. IntendedFor is deprecated; the B0FieldIdentifier/B0FieldSource links are used exclusively and the IntendedFor entries are ignored.


**2. It builds the same wall as `B0Field*` curation** (section 5): a series
named by a fieldmap's `IntendedFor` counts as linked, and linked series are
never pooled with unlinked ones by the reverse-PE heuristic. Here the
fieldmap is intended only for the AP series, so the unlinked PA series
cannot pair with AP and goes uncorrected:


In [19]:
ap = scan('sub-01_dir-AP_dwi.nii.gz', pe_dir='j-')
pa = scan('sub-01_dir-PA_dwi.nii.gz', pe_dir='j')
fmap = scan(
    'sub-01_dir-PA_epi.nii.gz',
    folder='fmap',
    pe_dir='j',
    intended_for=ap.path,  # links AP only; PA is left unlinked
)
g = group(ap, pa, fmap)
print(report_text(g))

DWI grouping for sub-01

Output "sub-01_dir-AP" (MultipartID auto+concat+0 [inferred]): 1 series
  Distortion group sub-01_dir-AP (PE j-, TRT 0.05s):
    - sub-01_dir-AP_dwi.nii.gz
    corrected by: auto+fmap+sub-01_dir-PA_epi [intendedfor]

Output "sub-01_dir-PA" (MultipartID auto+concat+1 [inferred]): 1 series
  Distortion group sub-01_dir-PA (PE j, TRT 0.05s):
    - sub-01_dir-PA_dwi.nii.gz
    corrected by: nothing (no fieldmap found)

Fieldmap estimations:
  auto+fmap+sub-01_dir-PA_epi [intendedfor]: PEPOLAR (reverse phase-encoding)
    - sub-01_dir-AP_dwi.nii.gz
    - sub-01_dir-PA_epi.nii.gz
    phase encoding axes: j (bidirectional)

Notes:
  WARNING [reverse-pe-not-pooled]: sub-01_dir-PA_dwi.nii.gz: the only reverse phase-encoded series on this axis are already linked to fieldmaps by curated metadata (B0FieldIdentifier/B0FieldSource or IntendedFor), and linked series are never pooled with unlinked ones. Curate the metadata on every series (or on none) to control this.



**3. A fieldmap with neither field is never used.** It draws an
`unlinked-fmap` warning and is ignored entirely.

## Try it on your own data

Everything above runs from a real BIDS directory too — without processing
anything:

```bash
python -m qsiprep.grouping /path/to/bids --html grouping.html
```

prints the grouping report plus all three backend previews, and writes the
explanatory page you saw throughout this notebook (one per subject). If a
grouping decision surprises you, the provenance chip tells you which sidecar
field to set — `B0FieldIdentifier`, `B0FieldSource`, or `MultipartID` — to make
your intent explicit. Curate it once, and QSIPrep (and every other BIDS app)
will stop guessing.
